In [ ]:
%pip install datasets translatepy tqdm

In [2]:
import os
import json
import time
import random
import logging
from datasets import load_dataset, Dataset, DatasetDict
from translatepy import Translator
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

# Создаём переводчик
yandex = YandexTranslate()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Тест переводчика
test_text = "Hello, world!"
result = yandex.translate(test_text, "ru")
print(f"Original: {test_text}")
print(f"Translated: {result}")

Original: Hello, world!
Translated: Привет, мир!


In [3]:
SOURCE_REPO_ID = "DeepPavlov/canard"
LOCAL_SAVE_PATH = "./canard_ru"

# Файлы прогресса для каждой конфигурации
PROGRESS_FILE_CORPUS = "translated_canard_corpus_progress.jsonl"
PROGRESS_FILE_QUERIES = "translated_canard_queries_progress.jsonl"
PROGRESS_FILE_QRELS = "translated_canard_qrels_progress.jsonl"
CACHE_FILE = "translation_cache.jsonl"

In [4]:
def load_cache() -> dict[str, str]:
    cache: dict[str, str] = {}
    if not os.path.exists(CACHE_FILE):
        return cache
    try:
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj: dict[str, str] = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except Exception:
                    continue
    except Exception as e:
        logging.error(f"Failed to load cache: {e}")
    return cache

def append_cache(text: str, translation: str) -> None:
    try:
        with open(CACHE_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")
    except Exception as e:
        logging.error(f"Failed to append cache: {e}")

translation_cache: dict[str, str] = load_cache()

In [12]:
def translate_with_yandex(text: str, retries: int = 3, delay: int = 5) -> tuple[str, bool]:
    """
    Возвращает (перевод, успех_или_нет)
    """
    if not isinstance(text, str) or text.strip() == "":
        return "", True
    
    if text in translation_cache:
        return translation_cache[text], True
    
    for attempt in range(retries):
        try:
            time.sleep(0.5)
            result = yandex.translate(text, "ru")
            
            # Извлекаем текст из результата
            if hasattr(result, 'result'):
                translated_text = str(result.result)
            else:
                translated_text = str(result)
            
            translation_cache[text] = translated_text
            append_cache(text, translated_text)
            return translated_text, True
            
        except Exception as e:
            logging.warning(f"Translation error for text: '{text[:50]}...'. Attempt {attempt + 1}/{retries}. Error: {e}")
            if attempt < retries - 1:
                time.sleep(delay)
    
    # Если все попытки не удались
    logging.error(f"Failed to translate after {retries} attempts: '{text[:50]}...'")
    return "", False

In [23]:
def process_and_translate_split_with_history(split_name, source_dataset_split, progress_file, text_field='text', history_field='history'):
    """
    Функция для перевода queries с учетом history
    """
    translated_records = []
    failed_indices = []
    
    # Загружаем уже переведенные записи
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line)
                    if record.get('_failed', False):
                        failed_indices.append(record.get('_index', -1))
                    else:
                        translated_records.append(record)
                except:
                    continue
        logging.info(f"Resuming {split_name}. Found {len(translated_records)} successful, {len(failed_indices)} failed records.")
    
    successful_indices = {record['_index'] for record in translated_records if '_index' in record}
    start_index = len(translated_records) + len(failed_indices)
    
    if start_index < len(source_dataset_split):
        logging.info(f"Starting translation for '{split_name}' from index {start_index}...")
        
        with open(progress_file, "a", encoding="utf-8") as f:
            pbar = tqdm(
                enumerate(source_dataset_split.select(range(start_index, len(source_dataset_split)))),
                desc=f"Translating {split_name}",
                total=len(source_dataset_split) - start_index
            )
            
            batch_count = 0
            for idx, example in pbar:
                global_idx = start_index + idx
                
                if global_idx in successful_indices:
                    continue
                
                # Переводим text
                original_text = example[text_field]
                translated_text, success_text = translate_with_yandex(original_text)
                
                # Переводим history, если есть
                translated_history = None
                success_history = True
                if history_field in example and example[history_field]:
                    original_history = example[history_field]
                    translated_history, success_history = translate_with_yandex(original_history)
                
                success = success_text and success_history
                
                new_record = {
                    '_index': global_idx,
                    '_failed': not success,
                    text_field: original_text,
                    f"{text_field}_ru": translated_text if success_text else None,
                }
                
                # Добавляем history
                if history_field in example:
                    new_record[history_field] = example[history_field]
                    new_record[f"{history_field}_ru"] = translated_history if success_history else None
                
                # Копируем остальные поля
                for key, value in example.items():
                    if key not in [text_field, history_field]:
                        new_record[key] = value
                
                f.write(json.dumps(new_record, ensure_ascii=False) + "\n")
                f.flush()
                
                if success:
                    translated_records.append(new_record)
                else:
                    failed_indices.append(global_idx)
                
                pbar.set_postfix({'success': len(translated_records), 'failed': len(failed_indices)})
                
                batch_count += 1
                if batch_count % 10 == 0:
                    pause = random.uniform(5, 10)
                    logging.info(f"Taking a short break of {pause:.1f} seconds after {batch_count} requests...")
                    time.sleep(pause)
    
    # Формируем финальный датасет
    all_successful = []
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line)
                    if not record.get('_failed', False):
                        record.pop('_index', None)
                        record.pop('_failed', None)
                        all_successful.append(record)
                except:
                    continue
    
    if not all_successful:
        logging.error(f"No records were successfully translated for {split_name}. Aborting.")
        return None
    
    total = len(source_dataset_split)
    successful_count = len(all_successful)
    failed_count = total - successful_count
    logging.info(f"{split_name}: {successful_count}/{total} translated successfully ({failed_count} failed)")
    
    return Dataset.from_list(all_successful)

In [24]:
# Функция для перезапуска только неудачных записей
def retry_failed_translations(split_name, source_dataset_split, progress_file, text_field='text'):
    """
    Перезапускает только те записи, которые не перевелись в прошлый раз
    """
    failed_records = []
    
    if not os.path.exists(progress_file):
        logging.error(f"Progress file {progress_file} not found")
        return None
    
    # Загружаем неудачные записи
    with open(progress_file, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            if record.get('_failed', False):
                failed_records.append(record)
    
    if not failed_records:
        logging.info(f"No failed records found for {split_name}")
        return None
    
    logging.info(f"Retrying {len(failed_records)} failed translations for {split_name}...")
    
    # Создаем временный файл для новых успешных переводов
    temp_progress_file = progress_file + ".retry"
    successful_retries = 0
    
    with open(temp_progress_file, "w", encoding="utf-8") as f_out:
        # Сначала записываем все успешные записи из исходного файла
        with open(progress_file, "r", encoding="utf-8") as f_in:
            for line in f_in:
                record = json.loads(line)
                if not record.get('_failed', False):
                    f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
        
        # Пробуем перевести неудачные записи заново
        for record in tqdm(failed_records, desc=f"Retrying {split_name}"):
            original_text = record[text_field]
            translated_text, success = translate_with_yandex(original_text)
            
            if success:
                new_record = {
                    '_index': record['_index'],
                    '_failed': False,
                    text_field: original_text,
                    f"{text_field}_ru": translated_text
                }
                # Копируем остальные поля
                for key, value in record.items():
                    if key not in ['_index', '_failed', text_field, f"{text_field}_ru"]:
                        new_record[key] = value
                
                f_out.write(json.dumps(new_record, ensure_ascii=False) + "\n")
                successful_retries += 1
            else:
                # Оставляем как есть
                f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
    
    # Заменяем старый файл новым
    os.replace(temp_progress_file, progress_file)
    
    logging.info(f"Retry complete: {successful_retries}/{len(failed_records)} succeeded")
    
    # Возвращаем обновленный датасет
    return process_and_translate_split(split_name, source_dataset_split, progress_file, text_field)

In [ ]:
logging.info(f"Loading source dataset '{SOURCE_REPO_ID}'...")
source_dataset = {
    'corpus': load_dataset(SOURCE_REPO_ID, 'corpus'),
    'queries': load_dataset(SOURCE_REPO_ID, 'queries'),
    'qrels': load_dataset(SOURCE_REPO_ID, 'qrels')
}

print("Source dataset loaded:")
for key, ds in source_dataset.items():
    print(f"\n{key.upper()}:")
    print(ds)
    if len(ds['train']) > 0:
        print(f"Example from {key} train:")
        print(ds['train'][0])

In [ ]:
# Переводим все сплиты корпуса
corpus_splits = {}
for split in ['train', 'validation', 'test']:
    progress_file = f"translated_canard_corpus_{split}.jsonl"
    corpus_splits[split] = process_and_translate_split(
        f"corpus/{split}", 
        source_dataset['corpus'][split], 
        progress_file, 
        'text'
    )
    
    if corpus_splits[split] is None:
        logging.error(f"Failed to translate corpus/{split}")
        break

if all(v is not None for v in corpus_splits.values()):
    corpus_translated = DatasetDict(corpus_splits)
    logging.info("Corpus translation completed")
else:
    corpus_translated = None

In [ ]:
# Переводим все сплиты запросов
queries_splits = {}
for split in ['train', 'validation', 'test']:
    progress_file = f"translated_canard_queries_{split}.jsonl"
    queries_splits[split] = process_and_translate_split_with_history(
        f"queries/{split}", 
        source_dataset['queries'][split], 
        progress_file, 
        'text',
        'history'
    )
    
    if queries_splits[split] is None:
        logging.error(f"Failed to translate queries/{split}")
        break

if all(v is not None for v in queries_splits.values()):
    queries_translated = DatasetDict(queries_splits)
    logging.info("Queries translation completed")
else:
    queries_translated = None

In [ ]:
logging.info("Qrels doesn't contain text to translate, copying as is...")
qrels_translated = DatasetDict({
    split: source_dataset['qrels'][split] 
    for split in source_dataset['qrels'].keys()
})

In [ ]:
# После основного перевода, если были ошибки
for split in ['train', 'validation', 'test']:
    progress_file = f"translated_canard_corpus_{split}.jsonl"
    retry_failed_translations(
        f"corpus/{split}", 
        source_dataset['corpus'][split], 
        progress_file, 
        'text'
    )

In [ ]:
if corpus_translated is None or queries_translated is None:
    logging.error("One of the configurations failed to process. Halting.")
else:
    # Функция для проверки и заполнения недостающих полей
    def ensure_translation_fields(dataset_dict, has_history=False):
        """
        Проверяет наличие перевода и подставляет оригинал при необходимости
        """
        for split_name in dataset_dict.keys():
            split = dataset_dict[split_name]
            
            # Проверяем и исправляем каждую запись
            def fix_record(example):
                # Для text
                if 'text_ru' not in example or example['text_ru'] is None:
                    example['text_ru'] = example.get('text', '')
                    logging.debug(f"Missing translation for text in {split_name}, using original")
                
                # Для history (если есть)
                if has_history and 'history' in example:
                    if 'history_ru' not in example or example['history_ru'] is None:
                        example['history_ru'] = example.get('history', '')
                        logging.debug(f"Missing translation for history in {split_name}, using original")
                
                return example
            
            # Применяем исправления ко всем записям
            dataset_dict[split_name] = split.map(fix_record)
        
        return dataset_dict
    
    # Применяем проверку к корпусу и запросам
    logging.info("Checking and fixing missing translations...")
    corpus_translated = ensure_translation_fields(corpus_translated, has_history=False)
    queries_translated = ensure_translation_fields(queries_translated, has_history=True)
    
    final_dataset = DatasetDict({
        "corpus": corpus_translated,
        "queries": queries_translated,
        "qrels": qrels_translated
    })
    
    print("\nFinal translated dataset structure:")
    for key, ds in final_dataset.items():
        print(f"\n{key.upper()}:")
        print(ds)
        
        # Статистика по переводам
        if key in ['corpus', 'queries']:
            total = len(ds['train'])
            translated_count = 0
            for split in ds.keys():
                for example in ds[split]:
                    if 'text_ru' in example and example['text_ru'] != example.get('text', ''):
                        translated_count += 1
            print(f"  Translation coverage: {translated_count}/{total * len(ds.keys())} fields")
    
    print("\nExample from translated corpus (train):")
    if len(final_dataset["corpus"]["train"]) > 0:
        example = final_dataset["corpus"]["train"][0]
        print(f"ID: {example.get('id', 'N/A')}")
        print(f"Original: {example.get('text', 'N/A')}")
        print(f"Translated: {example.get('text_ru', 'N/A')}")
        if example.get('text_ru') == example.get('text'):
            print("  ⚠️ Using original text (translation missing)")
    
    print("\nExample from translated queries (train):")
    if len(final_dataset["queries"]["train"]) > 0:
        example = final_dataset["queries"]["train"][0]
        print(f"ID: {example.get('id', 'N/A')}")
        print(f"Original text: {example.get('text', 'N/A')}")
        print(f"Translated text: {example.get('text_ru', 'N/A')}")
        if example.get('text_ru') == example.get('text'):
            print("  ⚠️ Using original text (translation missing)")
        
        if 'history' in example:
            print(f"Original history: {example.get('history', 'N/A')[:100]}...")
            print(f"Translated history: {example.get('history_ru', 'N/A')[:100]}...")
            if example.get('history_ru') == example.get('history'):
                print("  ⚠️ Using original history (translation missing)")
    
    logging.info(f"Saving translated dataset locally to '{LOCAL_SAVE_PATH}'...")
    final_dataset.save_to_disk(LOCAL_SAVE_PATH)
    print(f"Dataset saved to {LOCAL_SAVE_PATH}")
    
    # Сохраняем также в формате JSON для удобства просмотра
    logging.info("Saving sample as JSON for inspection...")
    sample_data = {
        "corpus_train_sample": final_dataset["corpus"]["train"][:5],
        "queries_train_sample": final_dataset["queries"]["train"][:5]
    }
    with open(f"{LOCAL_SAVE_PATH}_sample.json", "w", encoding="utf-8") as f:
        json.dump(sample_data, f, ensure_ascii=False, indent=2)
    print(f"Sample saved to {LOCAL_SAVE_PATH}_sample.json")

In [ ]:
def new_func():
    if corpus_translated is None or queries_translated is None:
        logging.error("One of the configurations failed to process. Halting.")
    else:
        final_dataset = DatasetDict({
        "corpus": corpus_translated,
        "queries": queries_translated,
        "qrels": qrels_translated
    })
    
        print("\nFinal translated dataset structure:")
        for key, ds in final_dataset.items():
            print(f"\n{key.upper()}:")
            print(ds)
    
        print("\nExample from translated corpus (train):")
        if len(final_dataset["corpus"]["train"]) > 0:
            example = final_dataset["corpus"]["train"][0]
            print(f"Original: {example.get('text', 'N/A')}")
            print(f"Translated: {example.get('text_ru', 'N/A')}")
    
        print("\nExample from translated queries (train):")
        if len(final_dataset["queries"]["train"]) > 0:
            example = final_dataset["queries"]["train"][0]
            print(f"Original: {example.get('text', 'N/A')}")
            print(f"Translated: {example.get('text_ru', 'N/A')}")
    
        logging.info(f"Saving translated dataset locally to '{LOCAL_SAVE_PATH}'...")
        final_dataset.save_to_disk(LOCAL_SAVE_PATH)
        print(f"Dataset saved to {LOCAL_SAVE_PATH}")

new_func()

In [ ]:
def check_translation_status():
    """
    Проверяет статус всех переводов
    """
    print("=" * 60)
    print("TRANSLATION STATUS REPORT")
    print("=" * 60)
    
    # Проверка корпуса
    print("\nCORPUS:")
    for split in ['train', 'validation', 'test']:
        progress_file = f"translated_canard_corpus_{split}.jsonl"
        if os.path.exists(progress_file):
            successful = 0
            failed = 0
            with open(progress_file, "r", encoding="utf-8") as f:
                for line in f:
                    record = json.loads(line)
                    if record.get('_failed', False):
                        failed += 1
                    else:
                        successful += 1
            
            total = successful + failed
            progress = (successful / total * 100) if total > 0 else 0
            print(f"  {split}: {successful}/{total} ({progress:.1f}%) - {failed} failed")
        else:
            print(f"  {split}: No progress file found")
    
    # Проверка запросов
    print("\nQUERIES:")
    for split in ['train', 'validation', 'test']:
        progress_file = f"translated_canard_queries_{split}.jsonl"
        if os.path.exists(progress_file):
            successful = 0
            failed = 0
            with open(progress_file, "r", encoding="utf-8") as f:
                for line in f:
                    record = json.loads(line)
                    if record.get('_failed', False):
                        failed += 1
                    else:
                        successful += 1
            
            total = successful + failed
            progress = (successful / total * 100) if total > 0 else 0
            print(f"  {split}: {successful}/{total} ({progress:.1f}%) - {failed} failed")
        else:
            print(f"  {split}: No progress file found")
    
    print("\n" + "=" * 60)

# Проверяем статус
check_translation_status()

In [ ]:
from datasets import load_from_disk, DatasetDict
from huggingface_hub import login

login(token="YOUR_HF_Token")

base = "/Users/polinakremneva/code/deep-pavlov/canard_ru"

# Загружаем каждую конфигурацию как отдельный DatasetDict
corpus = load_from_disk(f"{base}/corpus")
queries = load_from_disk(f"{base}/queries")
qrels = load_from_disk(f"{base}/qrels")

# Загружаем каждую конфигурацию отдельно на Hub
print("Uploading corpus...")
corpus.push_to_hub("DeepPavlov/canard_ru", config_name="corpus", private=False)

print("Uploading queries...")
queries.push_to_hub("DeepPavlov/canard_ru", config_name="queries", private=False)

print("Uploading qrels...")
qrels.push_to_hub("DeepPavlov/canard_ru", config_name="qrels", private=False)

print("✅ Done! https://huggingface.co/datasets/DeepPavlov/canard_ru")